In [3]:
# =====================
# 1. 라이브러리 임포트
# =====================
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from rasterio.sample import sample_gen
import requests
from shapely.geometry import Point

# =====================
# 2. 강원도 AWS 지점 목록
# =====================
AUTH_KEY = "EzBXCbK3SQGwVwmyt6kBfQ"

stn_url = "https://apihub.kma.go.kr/api/typ01/url/stn_inf.php"
params = {
    "inf": "AWS", "stn": "", "tm": "202401010000",
    "help": "0", "authKey": AUTH_KEY
}
res = requests.get(stn_url, params=params)
lines = res.text.split("\n")

stations = []
for line in lines:
    if line.startswith("#") or line.strip() == "" or "7777" in line:
        continue
    parts = line.split()
    if len(parts) >= 9:
        try:
            stations.append({
                "stn_id": parts[0],
                "lon": float(parts[1]),
                "lat": float(parts[2]),
                "name": parts[8]
            })
        except:
            continue

df_stn = pd.DataFrame(stations)
gangwon = df_stn[
    (df_stn["lat"] >= 37.0) & (df_stn["lat"] <= 38.6) &
    (df_stn["lon"] >= 127.5) & (df_stn["lon"] <= 129.4)
].reset_index(drop=True)

print(f"강원도 AWS: {len(gangwon)}개")

# =====================
# 3. NDVI 추출
# =====================
with rasterio.open("../data/raw/NDVI_Gangwon_scaled.tif") as src:
    coords = [(row["lon"], row["lat"]) for _, row in gangwon.iterrows()]
    ndvi_values = list(sample_gen(src, coords))

gangwon["ndvi_mean"] = [v[0] for v in ndvi_values]
print(f"NDVI 추출 완료: {gangwon['ndvi_mean'].notna().sum()}개 지점")

# =====================
# 4. 산림입지도 공간 조인
# =====================
gdf = gpd.read_file("../data/raw/산림입지도/산림입지도.shp", encoding="cp949")
gdf = gdf.to_crs(epsg=4326)

key_cols = [
    "SLTP_CD", "TPGRP_TPCD", "CLZN_CD", "PRRCK_LARG",
    "SOIL_DRNGE", "LOCTN_GRDN", "ALTTD_CD", "WASH_CD",
    "SLANT_TYP", "EIGHT_CD", "WIND_EXDGR", "WTEFF_DGR",
    "VLDTY_SLDP", "KOFTR_CD", "AVRG_FRAG", "geometry"
]
gdf_slim = gdf[key_cols].copy()

gdf_aws = gpd.GeoDataFrame(
    gangwon,
    geometry=[Point(row["lon"], row["lat"]) for _, row in gangwon.iterrows()],
    crs="EPSG:4326"
)

gdf_joined = gpd.sjoin_nearest(gdf_aws, gdf_slim, how="left", max_distance=0.1)

# =====================
# 5. 코드값 변환 및 저장
# =====================
soil_map = {
    "01":"갈색건조", "02":"갈색약건", "03":"갈색적윤", "04":"갈색약습",
    "05":"적색계갈색건조", "06":"적색계갈색약건", "07":"적색건조", "08":"적색약건",
    "10":"암적색건조", "11":"암적색약건", "12":"암적색적윤",
    "24":"약침식토양", "25":"강침식토양", "27":"미숙토양", "28":"암쇄토양"
}
topo_map = {
    "01":"산정", "02":"산능선", "03":"산복", "04":"산곡",
    "05":"산록", "06":"계곡", "07":"구릉지", "08":"능선", "10":"평지"
}
climate_map = {"1":"온대북부", "2":"온대중부", "3":"온대남부", "4":"난대"}
rock_map = {"1":"화성암", "2":"퇴적암", "3":"변성암"}
drain_map = {"1":"불량", "2":"보통", "3":"양호", "4":"매우양호"}
wash_map = {"1":"없음", "2":"있음", "3":"많음"}
slant_map = {"1":"상승사면", "2":"평행사면", "3":"하강사면"}
wind_map = {"1":"노출", "2":"보통", "3":"보호"}
weather_map = {"01":"상", "02":"중", "03":"하"}
tree_map = {
    "11005":"강원소나무", "11006":"중부소나무", "11017":"낙엽송",
    "11019":"잣나무", "12017":"상수리나무", "12018":"신갈나무",
    "11002":"리기다소나무", "12999":"기타활엽수", "11999":"기타침엽수"
}
direction_map = {
    "100":"동", "200":"서", "300":"남", "400":"북",
    "310":"남동", "320":"남서", "410":"북동", "420":"북서",
    "330":"남남동", "340":"남동동", "350":"남남서", "360":"남서서",
    "430":"북북동", "440":"북동동", "450":"북북서", "460":"북서서",
    "999":"평탄"
}
gdf_result = gdf_joined[[
    "stn_id", "name", "lat", "lon", "ndvi_mean",
    "SLTP_CD", "TPGRP_TPCD", "CLZN_CD", "PRRCK_LARG",
    "SOIL_DRNGE", "LOCTN_GRDN", "ALTTD_CD", "WASH_CD",
    "SLANT_TYP", "EIGHT_CD", "WIND_EXDGR", "WTEFF_DGR",
    "VLDTY_SLDP", "KOFTR_CD", "AVRG_FRAG"
]].copy()

gdf_result["토양형"] = gdf_result["SLTP_CD"].map(soil_map)
gdf_result["지형"] = gdf_result["TPGRP_TPCD"].map(topo_map)
gdf_result["기후대"] = gdf_result["CLZN_CD"].astype(str).map(climate_map)
gdf_result["모암대"] = gdf_result["PRRCK_LARG"].astype(str).map(rock_map)
gdf_result["배수"] = gdf_result["SOIL_DRNGE"].astype(str).map(drain_map)
gdf_result["침식"] = gdf_result["WASH_CD"].astype(str).map(wash_map)
gdf_result["경사형"] = gdf_result["SLANT_TYP"].astype(str).map(slant_map)
gdf_result["사면방위"] = gdf_result["EIGHT_CD"].astype(str).map(direction_map)
gdf_result["바람노출"] = gdf_result["WIND_EXDGR"].astype(str).map(wind_map)
gdf_result["풍화정도"] = gdf_result["WTEFF_DGR"].astype(str).map(weather_map)
gdf_result["수종"] = gdf_result["KOFTR_CD"].astype(str).map(tree_map)

gdf_result.to_csv("../data/processed/gangwon_aws_soil.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료: {gdf_result.shape}")
print(gdf_result[["stn_id", "name", "토양형", "지형", "기후대", "모암대", "배수", "침식", "경사형", "사면방위", "바람노출", "풍화정도", "수종"]].head(10))

강원도 AWS: 106개
NDVI 추출 완료: 106개 지점
저장 완료: (106, 31)
  stn_id   name      토양형   지형   기후대  모암대    배수  침식   경사형 사면방위 바람노출 풍화정도     수종
0     90     속초     갈색약건   산복  온대중부  화성암    보통  없음  평행사면   남동   보통    중    NaN
1     92  양양(공)     갈색약건   산록  온대중부  화성암    보통  없음  평행사면   북서   보통    중    NaN
2     93    북춘천     갈색약건   산록  온대중부  화성암    보통  없음  평행사면   남서   보통    중    NaN
3    100    대관령     갈색약건   산록  온대북부  변성암    양호  없음  평행사면   북서   보통    중  강원소나무
4    101     춘천     갈색적윤   산록  온대중부  변성암    양호  없음  하강사면   남서   보통    상    NaN
5    104    북강릉  적색계갈색건조   산복  온대중부  화성암    양호  없음  평행사면   남동   보통    중  강원소나무
6    105     강릉  적색계갈색약건  구릉지  온대중부  화성암  매우양호  없음  평행사면   북동   보통    상  강원소나무
7    106     동해    암적색건조   산정  온대남부  퇴적암    보통  있음  평행사면   남동   보통    상  강원소나무
8    114     원주     갈색약건   산복  온대중부  화성암    보통  없음  평행사면   북동   보통    중  상수리나무
9    121     영월    암적색약건  구릉지  온대중부  퇴적암    보통  있음  상승사면   북동   노출    중  강원소나무


C:\Users\ssoos\Desktop\Forest_recovery\venv\Lib\site-packages\geopandas\array.py:411: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


In [4]:
# 산사태위험지도 AWS 지점별 등급 추출
with rasterio.open("../data/raw/산사태위험지도/산사태위험지도.tif") as src:
    print(f"좌표계: {src.crs}")
    print(f"크기: {src.width} x {src.height}")
    print(f"범위: {src.bounds}")
    
    # 좌표계 확인 후 AWS 지점 좌표 변환 필요할 수 있음
    coords = [(row["lon"], row["lat"]) for _, row in gangwon.iterrows()]
    landslide_values = list(sample_gen(src, coords))

gangwon["산사태등급"] = [v[0] for v in landslide_values]
print(f"\n산사태등급 분포:")
print(gangwon["산사태등급"].value_counts().sort_index())

좌표계: PROJCS["KGD2002 / Central Belt",GEOGCS["KGD2002",DATUM["Korean_Geodetic_Datum_2002",SPHEROID["GRS 1980",6378137,298.257222101004,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6737"]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4737"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",38],PARAMETER["central_meridian",127],PARAMETER["scale_factor",1],PARAMETER["false_easting",200000],PARAMETER["false_northing",500000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5181"]]
크기: 20210 x 17723
범위: BoundingBox(left=207990.78376534837, bottom=392770.61906786414, right=410090.7837653484, top=570000.6190678641)

산사태등급 분포:
산사태등급
127    106
Name: count, dtype: int64


In [5]:
from pyproj import Transformer

# WGS84 → EPSG:5181 변환
transformer = Transformer.from_crs("EPSG:4326", "EPSG:5181", always_xy=True)

with rasterio.open("../data/raw/산사태위험지도/산사태위험지도.tif") as src:
    # 좌표 변환 (경도, 위도 → 투영좌표)
    coords_proj = [
        transformer.transform(row["lon"], row["lat"]) 
        for _, row in gangwon.iterrows()
    ]
    landslide_values = list(sample_gen(src, coords_proj))

gangwon["산사태등급"] = [v[0] for v in landslide_values]
print(f"산사태등급 분포:")
print(gangwon["산사태등급"].value_counts().sort_index())

산사태등급 분포:
산사태등급
1        1
2        1
3        1
5        2
127    101
Name: count, dtype: int64


In [6]:
# 127(nodata) → NaN 처리
gangwon["산사태등급"] = gangwon["산사태등급"].replace(127, np.nan)

landslide_map = {1: "1등급(매우위험)", 2: "2등급(위험)", 3: "3등급(보통)", 
                 4: "4등급(낮음)", 5: "5등급(매우낮음)"}
gangwon["산사태위험도"] = gangwon["산사태등급"].map(landslide_map)

print(gangwon["산사태위험도"].value_counts())

# gangwon_aws_soil.csv에 병합해서 저장
df_soil = pd.read_csv("../data/processed/gangwon_aws_soil.csv", encoding="utf-8-sig")
df_soil = df_soil.merge(
    gangwon[["stn_id", "산사태등급", "산사태위험도"]], 
    on="stn_id", how="left"
)
df_soil.to_csv("../data/processed/gangwon_aws_soil.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료: {df_soil.shape}")

산사태위험도
5등급(매우낮음)    2
3등급(보통)      1
2등급(위험)      1
1등급(매우위험)    1
Name: count, dtype: int64


ValueError: You are trying to merge on int64 and str columns for key 'stn_id'. If you wish to proceed you should use pd.concat